# Weather Data Ingestion - NSW Regional Coverage

**Source:** Open-Meteo API (https://api.open-meteo.com)  
**Target:** `mobility_ai.bronze.weather_raw`  
**Mode:** Overwrite  
**Coverage:** 20 weather stations across all NSW regions

**Regions Covered:**
* Metro Sydney and surrounds (Sydney, Central Coast, Blue Mountains, Wollongong)
* Hunter Region (Newcastle, Tamworth)
* North Coast (Byron Bay, Lismore, Grafton, Coffs Harbour, Port Macquarie)
* New England (Armidale)
* Central West (Orange, Bathurst, Dubbo)
* Far West (Broken Hill)
* Riverina (Wagga Wagga, Albury)
* South Coast (Bega)
* Southern Tablelands (Goulburn)

**Purpose:** Provides real-time weather conditions across NSW to help users find charging stations and fuel stations considering current weather conditions.

In [0]:
import requests
import pandas as pd
from datetime import datetime

# Configuration
TARGET_TABLE = "mobility_ai.bronze.weather_raw"
SOURCE_SYSTEM = "open-meteo"
API_TIMEOUT = 10

# NSW regional locations for comprehensive weather coverage
nsw_locations = [
    {"station_id": "SYD001", "station_name": "Sydney", "lat": -33.8688, "lon": 151.2093},
    {"station_id": "NEW001", "station_name": "Newcastle", "lat": -32.9283, "lon": 151.7817},
    {"station_id": "WOL001", "station_name": "Wollongong", "lat": -34.4278, "lon": 150.8931},
    {"station_id": "CEN001", "station_name": "Central Coast", "lat": -33.3094, "lon": 151.3814},
    {"station_id": "BLU001", "station_name": "Blue Mountains", "lat": -33.7122, "lon": 150.3106},
    {"station_id": "COF001", "station_name": "Coffs Harbour", "lat": -30.2963, "lon": 153.1158},
    {"station_id": "POR001", "station_name": "Port Macquarie", "lat": -31.4331, "lon": 152.9083},
    {"station_id": "BYR001", "station_name": "Byron Bay", "lat": -28.6436, "lon": 153.6122},
    {"station_id": "TAM001", "station_name": "Tamworth", "lat": -31.0906, "lon": 150.9292},
    {"station_id": "ARM001", "station_name": "Armidale", "lat": -30.5139, "lon": 151.6686},
    {"station_id": "ORA001", "station_name": "Orange", "lat": -33.2839, "lon": 149.0995},
    {"station_id": "BAT001", "station_name": "Bathurst", "lat": -33.4194, "lon": 149.5781},
    {"station_id": "DUB001", "station_name": "Dubbo", "lat": -32.2569, "lon": 148.6011},
    {"station_id": "BRO001", "station_name": "Broken Hill", "lat": -31.9505, "lon": 141.4333},
    {"station_id": "WAG001", "station_name": "Wagga Wagga", "lat": -35.1082, "lon": 147.3598},
    {"station_id": "ALB001", "station_name": "Albury", "lat": -36.0737, "lon": 146.9135},
    {"station_id": "BEG001", "station_name": "Bega", "lat": -36.6742, "lon": 149.8414},
    {"station_id": "GOU001", "station_name": "Goulburn", "lat": -34.7531, "lon": 149.7181},
    {"station_id": "LIS001", "station_name": "Lismore", "lat": -28.8142, "lon": 153.2789},
    {"station_id": "GRA001", "station_name": "Grafton", "lat": -29.6886, "lon": 152.9347}
]

In [0]:
weather_records = []

for location in nsw_locations:
    try:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={location['lat']}&longitude={location['lon']}&current_weather=true"
        response = requests.get(url, timeout=API_TIMEOUT)
        response.raise_for_status()
        
        data = response.json()
        weather = data["current_weather"]
        
        weather_records.append({
            "station_id": location["station_id"],
            "station_name": location["station_name"],
            "location_lat": location["lat"],
            "location_lon": location["lon"],
            "timestamp": pd.to_datetime(weather["time"]),
            "temperature_celsius": weather["temperature"],
            "humidity_percent": None,  # Not available in current_weather endpoint
            "precipitation_mm": None,  # Not available in current_weather endpoint
            "wind_speed_kmh": weather["windspeed"],
            "weather_condition": str(weather["weathercode"]),
            "ingestion_timestamp": pd.Timestamp.now(),
            "source_system": SOURCE_SYSTEM
        })
        
    except Exception as e:
        print(f"Warning: Failed to fetch weather for {location['station_name']}: {str(e)}")
        continue

df = pd.DataFrame(weather_records)
spark_df = spark.createDataFrame(df)

print(f"Successfully fetched weather data for {len(weather_records)} NSW locations")

In [0]:
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TARGET_TABLE)

print(f"Written {len(weather_records)} weather records to {TARGET_TABLE}")

In [0]:
validation_df = spark.table(TARGET_TABLE)
record_count = validation_df.count()

print(f"Validation: {record_count} records in {TARGET_TABLE}")
display(validation_df.limit(10))